# Behind the pipeline

In [1]:
from transformers import pipeline

classifier = pipeline("sentiment-analysis")

classifier(
    [
        "I've been waiting for a HuggingFace course my whole life.",
        "I hate this so much!",
    ]
)

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use mps:0


[{'label': 'POSITIVE', 'score': 0.9598050713539124},
 {'label': 'NEGATIVE', 'score': 0.9994558691978455}]

This pipeline groups together three steps:

1. Preprocessing
2. Passing the inputs through the model
3. Post processing

## Preprocessing with a tokenizer

Like other neural networks, Transormer models can't process raw text directly, so the first step of our pipeline is to convert the text inputs into numbers that the model can make sense of.

To do this we use a `tokenizer`, which will be responsible for:
* Splitting the input into words, subwords, or symbols (like punctuation) that are called *tokens*
* Mapping each token to an integer
* Adding additional inputs that may be useful to the model

All this preprocessing needs to be done in exactly the same way as when the model was pretrained. To do this, we use the `AutoTokenizer` class and its `from_pretrained()` method. Using the checkpoint name for our model, it will automatically fetch the data associated with the model's tokenizer and cache it.

In [2]:
from transformers import AutoTokenizer

checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

Once we have the tokenizer, we can directly pass our sentences to it and we'll get back a dictionary that is ready to feed our model. The only thing left to do is to convert the list of input IDs to tensors.

Transformer models only accept `tensor` as input.
If this is your first time hearing about tensors, you can think of them as NumPy arrays instead. A NumPy array can be a scalar (0D),  a vector (1D), a matrix (2D), or have more dimensions.

To specify the type of tensors we want to get back (PyTorch or plain NumPy), we use the `return_tensors` argument:

In [3]:
raw_inputs = [
    "I've been waiting for a HuggingFace course my whole life.",
    "I hate this so much!",
]
inputs = tokenizer(raw_inputs, padding=True, truncation=True, return_tensors="pt")
print(inputs)

{'input_ids': tensor([[  101,  1045,  1005,  2310,  2042,  3403,  2005,  1037, 17662, 12172,
          2607,  2026,  2878,  2166,  1012,   102],
        [  101,  1045,  5223,  2023,  2061,  2172,   999,   102,     0,     0,
             0,     0,     0,     0,     0,     0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0]])}


The outut itself is a dictionary containing two keys, `input_ids` and `attention_mask`.

`input_ds` contains two rows of integers (one for each input sentence) that are the unique identifiers of the *tokens* in each sentence.

## Going through the model

We can download a pretrained model the same way we did with a tokenizer. Transformers provides an `AutoModel` class which also has a `from_pretrained()` method:

In [4]:
from transformers import AutoModel

checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
model = AutoModel.from_pretrained(checkpoint)

In the code snippet above, we downloaded the same checkpoint we used in our pipeline before and instantiated a model with it.

This architecture contains only the base Transformer module: given some inputs, it outputs what we'll call `hidden states`, also known as `features`. For each model input, we'll retrieve a high-dimensional vector representing the **contextual understanding of that input by the Transformer model**.

While these *hidden states* can be useful on their own, they're usually inputs to another part of the model, known as the `head`. Different tasks can be performed with the same architecture, but each of these tasks will have a different head associated with it.

### A high-dimensional vector?

The vector ouput by the Transformer module is usually large. It generally has three dimensions:
* **Batch Size:** The number of sequences processed at a time (2 in our example).
* **Sequence length:** The length of the numerical representation of the sequence (16 in our example.)
* **Hidden size:** The vector dimension of each model input.

It is said to be "high dimensional" because of the last value. The hidden size can be very large (768 is common for smaller models, and in larger models this can reach 3072 or more).

In [5]:
outputs = model(**inputs)
print(outputs.last_hidden_state.shape)

torch.Size([2, 16, 768])


## Model heads: Making sense out of numbers

The model heads take the high-dimensional vector of hidden states as input and project them onto a different dimension. They are usually composed of one or a few linear layers.

The output of the Transformer model is sent directly to the model head to be processed.


For our example, we will need a model with a sequence classification head (to be able to classify the sentences as positive or negative). So, we won't actually use the `AutoModel` class, but `AutoModelForSequenceClassification`:

In [10]:
from transformers import AutoModelForSequenceClassification

checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)
outputs = model(**inputs)

Now if we look at the shape of our outputs, the dimensionality will be much lower: the model head takes as input the high-dimensional vectors we saw before, and outputs vectors containing two values (one per label):

In [11]:
print(outputs.logits)
print(outputs.logits.shape)

tensor([[-1.5607,  1.6123],
        [ 4.1692, -3.3464]], grad_fn=<AddmmBackward0>)
torch.Size([2, 2])


Since we have two sentences and two labels, the result we get from our model is of shape 2x2.

## Postprocessing the output

In [13]:
print(outputs.logits)

tensor([[-1.5607,  1.6123],
        [ 4.1692, -3.3464]], grad_fn=<AddmmBackward0>)


The model predicted `[-1.5607, 1.6123]` for the first sentence and `[4.1692, -3.3464]` for the second one. These are not probabilities but `logits`.

`Logits`: The raw, unnormalized scores outputted by the last layer of the model. To be converted to probabilities, they need to go through a SoftMax layer.

In [14]:
import torch

predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)
print(predictions)

tensor([[4.0195e-02, 9.5980e-01],
        [9.9946e-01, 5.4418e-04]], grad_fn=<SoftmaxBackward0>)


Now we can see that the model predicted `[0.0402, 0.9598]` for the first sentence and `[0.9995,0.0005]` for the second one. These are recognizable probability scores.

To get the labels corresponding to each position, we can inspect the `id2label` attribute of the model config.

In [16]:
model.config.id2label

{0: 'NEGATIVE', 1: 'POSITIVE'}

Now we can conclude that the model predicted the following:
* First sentence: NEGATIVE: 0.0402, POSITIVE: 0.9598
* Second sentence: NEGATIVE: 0.9995, POSITIVE: 0.0005